# Copilot Mode

Copilot mode is the autonomy level where the agent can *act* - but only with human approval before each consequential action is executed. The agent can read, search, analyse, and *also* prepare write operations: drafting an email to send, a file to write, a database record to update. But it hands every prepared action to the human to review and execute. The agent drives; the human approves.

This is the meaningful step above chat mode. Chat mode allows read-only tools but produces only text. Copilot mode adds access to write and action tools - it can construct and propose the full operation - while keeping a human in the loop as the final executor of every state change.

```
User Input
    │
    ▼
Agent (read-only + write/action tools available)
    │
    ├── search_web()         ← executes directly (read-only)
    ├── read_file()          ← executes directly (read-only)
    ├── query_database()     ← SELECT only, executes directly
    │
    ├── send_email()         ← prepared but NOT executed — handed to human
    ├── write_file()         ← prepared but NOT executed — handed to human
    └── update_record()      ← prepared but NOT executed — handed to human
    │
    ▼
Recommendation + Prepared Action (with exact parameters)
    │
    ▼
Human reviews and executes the write action
```

**Autonomy level: 1 — Supervised action**  
The agent's output is always a recommendation or a prepared action - never an executed write. Every consequential state change ends by handing control back to the human.

In [1]:
import os
from typing import Any
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, ToolMessage
from langchain_core.tools import tool

### Initialize the language model

In [2]:
llm = ChatOpenAI(model='gpt-4o-mini', api_key=os.getenv("OPENAI_API_KEY", "").strip(), temperature=0)

## Defining the tool set
Copilot mode exposes two categories of tools:

**Read-only tools** - execute directly, just as in chat mode. These gather information and have no side effects (web search, file reads, SELECT queries).

**Write/action tools** - available for the agent to *prepare* but never to *execute*. The agent uses them to construct the complete operation (with all parameters) and presents it to the human as an `ACTION REQUIRED` block. The human then executes it themselves.

The key implementation detail is that write tools are registered in the tool set so the model can reason about and prepare them - but an intercepting `CopilotModeEnforcer` blocks any attempt to actually run them, returning the prepared parameters for human review instead.

In a real deployment the read-only tools connect to live systems; the write tools may be stubs or real implementations gated behind the enforcer. Here we use mock implementations that return realistic responses so the agent's behaviour is fully observable.

In [3]:
# ── Read-only tools (execute directly) ───────────────────────────────────────

@tool
def search_knowledge_base(query: str) -> str:
    """Search the internal knowledge base for documentation and procedures.

    Args:
        query: The search query.

    Returns:
        Matching documentation snippets.
    """
    articles = {
        "deploy": (
            "Deployment checklist: (1) run tests, (2) build image, "
            "(3) push to registry, (4) update manifest, (5) smoke test."
        ),
        "rollback": (
            "Rollback procedure: revert image tag in manifest, "
            "drain pods, verify health endpoint returns 200."
        ),
        "database": (
            "DB access policy: read replicas are available for analytics. "
            "All writes must go through the primary with a reviewed migration."
        ),
        "security": (
            "Security policy: rotate credentials every 90 days, "
            "use Vault for secrets, never commit .env files."
        ),
    }
    for keyword, content in articles.items():
        if keyword in query.lower():
            return content
    return "No matching articles found. Try more specific terms."


@tool
def read_file(file_path: str) -> str:
    """Read the contents of a file. Read-only — does not modify the file.

    Args:
        file_path: Path to the file to read.

    Returns:
        File contents as a string.
    """
    files = {
        "config.yaml": "env: production\nreplicas: 3\nimage: app:v1.4.2\n",
        "README.md": "# App Service\nVersion 1.4.2 — see CHANGELOG for details.",
        "requirements.txt": "fastapi==0.104.1\nuvicorn==0.24.0\npydantic==2.5.0\n",
    }
    return files.get(file_path, f"File not found: {file_path}")


@tool
def query_database(sql_query: str) -> str:
    """Execute a read-only SELECT query against the database.

    Args:
        sql_query: A SQL SELECT query.

    Returns:
        Query results as a formatted string, or an error if not a SELECT.
    """
    if not sql_query.strip().upper().startswith("SELECT"):
        return (
            "ERROR: Only SELECT queries are permitted as read-only tool calls. "
            f"Received: {sql_query.split()[0].upper()}. "
            "Prepare an UPDATE/INSERT as an ACTION REQUIRED for human execution."
        )
    return (
        "user_id | plan   | status\n"
        "--------+--------+--------\n"
        "1042    | pro    | active\n"
        "1043    | basic  | active\n"
        "1044    | pro    | churned"
    )


# ── Write/action tools (intercepted — prepared for human, not executed) ───────

@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an email. Intercepted in Copilot Mode — prepared for human execution.

    Args:
        to: Recipient email address.
        subject: Email subject line.
        body: Email body text.

    Returns:
        Confirmation string (only reached in higher autonomy modes).
    """
    return f"Email sent to {to}: {subject}"


@tool
def write_file(file_path: str, content: str) -> str:
    """Write content to a file. Intercepted in Copilot Mode — prepared for human execution.

    Args:
        file_path: Path of the file to write.
        content: Content to write.

    Returns:
        Confirmation string (only reached in higher autonomy modes).
    """
    return f"File written: {file_path}"


@tool
def update_database(sql_query: str) -> str:
    """Execute a write query (INSERT/UPDATE/DELETE). Intercepted in Copilot Mode.

    Args:
        sql_query: The SQL write statement.

    Returns:
        Confirmation string (only reached in higher autonomy modes).
    """
    return f"Query executed: {sql_query}"


READ_ONLY_TOOLS = [search_knowledge_base, read_file, query_database]
WRITE_TOOLS = [send_email, write_file, update_database]
COPILOT_TOOLS = READ_ONLY_TOOLS + WRITE_TOOLS

print("Read-only tools (execute directly):", [t.name for t in READ_ONLY_TOOLS])
print("Write/action tools (prepared for human):", [t.name for t in WRITE_TOOLS])

Read-only tools (execute directly): ['search_knowledge_base', 'read_file', 'query_database']
Write/action tools (prepared for human): ['send_email', 'write_file', 'update_database']


## The write-action interceptor
Write tools are registered in the agent's tool set so the model can reason about them and prepare a complete, parameterised action. But a `CopilotModeEnforcer` intercepts every tool call at execution time: read-only calls pass through and execute; write calls are blocked and their prepared parameters are returned to the model as a tool result.

The model then incorporates the intercepted result into its final response - presenting the prepared write action in an `ACTION REQUIRED` section for the human to review and run.

This is *defence in depth*: the tool list says what the agent can reason about; the enforcer says what it can actually execute. Both layers together guarantee that the agent informs and prepares but never acts unilaterally.

In [5]:
class CopilotModeEnforcer:
    """Interceptor that blocks write tool execution in Copilot Mode.

    Read-only tool calls pass through and execute normally. Write/action tool calls are intercepted: 
    the prepared parameters are returned as a structured result so the model can present them to the human as an ACTION REQUIRED item.
    """

    WRITE_TOOLS = {
        "write_file", "send_email", "send_sms", "post_to_slack",
        "update_database", "delete_record", "insert_record",
        "deploy", "publish", "push_to_registry", "restart_service",
    }

    def check(self, tool_name: str, tool_args: dict) -> dict | None:
        """Check whether a tool call should execute or be intercepted.

        Args:
            tool_name: Name of the tool being called.
            tool_args: Arguments the agent prepared for the call.

        Returns:
            None if the call should execute (read-only); an interception dict
            if the call is a write operation that requires human execution.
        """
        if tool_name in self.WRITE_TOOLS:
            return {
                "intercepted": True,
                "tool": tool_name,
                "message": (
                    f"'{tool_name}' is a write operation. "
                    "Copilot Mode does not execute write actions automatically. "
                    "Present the parameters below to the human for their review and execution."
                ),
                "prepared_parameters": tool_args,
            }
        return None  # read-only — execute normally


enforcer = CopilotModeEnforcer()

# Read-only tool passes through
read_check = enforcer.check("search_knowledge_base", {"query": "deploy"})
# Write tool is intercepted
write_check = enforcer.check("send_email", {"to": "team@co.com", "subject": "Reminder", "body": "..."})

print("Read-only check (should be None):", read_check)
print()
print("Write tool check (intercepted):")
for k, v in write_check.items():
    print(f"  {k}: {v}")

Read-only check (should be None): None

Write tool check (intercepted):
  intercepted: True
  tool: send_email
  message: 'send_email' is a write operation. Copilot Mode does not execute write actions automatically. Present the parameters below to the human for their review and execution.
  prepared_parameters: {'to': 'team@co.com', 'subject': 'Reminder', 'body': '...'}


## The copilot agent
The copilot agent binds the LLM to the read-only tool set and runs an *agentic loop*: it invokes the model, checks whether any tool calls were requested, executes them, feeds the results back, and repeats until the model produces a final text response.

The system prompt does two important things: it tells the agent what tools it has and what it cannot do, and it specifies the output format - every response must end with an `ACTION REQUIRED` section that gives the human an unambiguous next step.

In [6]:
COPILOT_SYSTEM_PROMPT = """You are an AI assistant operating in Copilot Mode.

OPERATING CONSTRAINTS:
- Read-only tools (search_knowledge_base, read_file, query_database) execute automatically.
- Write/action tools (send_email, write_file, update_database) are available for you to
  PREPARE but will NOT be executed — they require human approval and execution.
- When a write tool call is intercepted, you will receive the prepared parameters back.
  Incorporate them into an ACTION REQUIRED section so the human knows exactly what to run.

YOUR ROLE:
- Gather needed information using read-only tools
- Analyse it thoroughly
- Prepare any required write actions with complete, correct parameters
- Always end your response with an explicit ACTION REQUIRED section

RESPONSE FORMAT:
FINDINGS:
[What you discovered from read-only tool calls]

RECOMMENDATION:
[What you suggest and why]

ACTION REQUIRED:
[Exact write operation(s) the human must execute, with all parameters ready to use]
"""


class CopilotAgent:
    """Agent that reads, analyses, and prepares actions — human executes write operations."""

    def __init__(self, llm, tools: list, enforcer: CopilotModeEnforcer):
        self.llm = llm.bind_tools(tools)
        self.tool_map = {t.name: t for t in tools}
        self.enforcer = enforcer

    def run(self, user_request: str) -> str:
        """Process a user request: execute read-only tools, intercept write tools.

        Args:
            user_request: The user's question or task description.

        Returns:
            The agent's final response including any ACTION REQUIRED items.
        """
        messages = [
            SystemMessage(content=COPILOT_SYSTEM_PROMPT),
            HumanMessage(content=user_request),
        ]

        while True:
            response = self.llm.invoke(messages)
            messages.append(response)

            if not response.tool_calls:
                return response.content

            for tc in response.tool_calls:
                intercepted = self.enforcer.check(tc["name"], tc["args"])

                if intercepted:
                    # Return the prepared parameters as the tool result
                    result = (
                        f"{intercepted['message']}\n"
                        f"Prepared parameters: {intercepted['prepared_parameters']}"
                    )
                else:
                    # Read-only tool — execute normally
                    result = self.tool_map[tc["name"]].invoke(tc["args"])

                messages.append(ToolMessage(content=str(result), tool_call_id=tc["id"]))


copilot = CopilotAgent(llm=llm, tools=COPILOT_TOOLS, enforcer=enforcer)
print("Copilot agent ready.")
print(f"  Read-only tools (auto-execute): {[t.name for t in READ_ONLY_TOOLS]}")
print(f"  Write tools (intercepted):      {[t.name for t in WRITE_TOOLS]}")

Copilot agent ready.
  Read-only tools (auto-execute): ['search_knowledge_base', 'read_file', 'query_database']
  Write tools (intercepted):      ['send_email', 'write_file', 'update_database']


## Demonstration 1: research and recommend
The most common copilot pattern is *research and recommend*: the user asks a question that requires gathering information from multiple sources, and the agent synthesises the findings into a recommendation with a clear action for the user to take.

Here the agent searches the knowledge base and reads a config file to assess deployment readiness, then hands the action back to the user.

In [7]:
print("=" * 65)
print("COPILOT MODE — Research and Recommend")
print("=" * 65)
print()

request = (
    "We want to deploy a new version of the app service. "
    "Check the current config and the deployment documentation, "
    "then tell me exactly what I need to do."
)
print(f"User request: {request}")
print()

recommendation = copilot.run(request)
print(recommendation)

COPILOT MODE — Research and Recommend

User request: We want to deploy a new version of the app service. Check the current config and the deployment documentation, then tell me exactly what I need to do.

FINDINGS:
1. The current configuration for the app service shows multiple user plans:
   - User ID 1042: Pro plan (active)
   - User ID 1043: Basic plan (active)
   - User ID 1044: Pro plan (churned)
   
2. The deployment documentation outlines the following checklist:
   - Run tests
   - Build image
   - Push to registry
   - Update manifest
   - Smoke test

RECOMMENDATION:
To deploy the new version of the app service, follow the deployment checklist step by step. Ensure that you run tests first to verify the new version's functionality, then proceed to build the image, push it to the registry, update the manifest, and finally perform a smoke test to confirm that everything is working as expected.

ACTION REQUIRED:
1. Run tests on the new version of the app service.
2. Build the imag

The response follows the template: findings from the tools, a concrete recommendation, and an explicit action list for the human. The agent never attempts to write the config or trigger the deployment itself.

## Demonstration 2: draft and review
A second common pattern is *draft and review*: the agent produces a complete artefact (email, document, SQL migration) for the human to review and submit. The agent does the drafting work; the human retains the send/submit action.

In [8]:
print("=" * 65)
print("COPILOT MODE — Draft and Review")
print("=" * 65)
print()

request = (
    "Look at the database security policy and draft an email I can send "
    "to the team reminding them of the credential rotation requirements."
)
print(f"User request: {request}")
print()

draft = copilot.run(request)
print(draft)

COPILOT MODE — Draft and Review

User request: Look at the database security policy and draft an email I can send to the team reminding them of the credential rotation requirements.

FINDINGS:
I was unable to locate specific information regarding the database security policy or the credential rotation requirements in the knowledge base. The only relevant information found pertains to the access policy for read replicas and write migrations.

RECOMMENDATION:
Since the specific details about the credential rotation requirements are not available, I suggest drafting a general reminder email to the team about the importance of credential rotation. This email can emphasize the need for security best practices and encourage team members to review their current credentials.

ACTION REQUIRED:
Please execute the following email preparation:

- **To:** [team@example.com] (replace with the actual team email)
- **Subject:** Reminder: Credential Rotation Requirements
- **Body:**
  ```
  Dear Team,


## Demonstration 3: enforcer intercepts a write attempt
This demonstration shows what happens when the model attempts a write operation. Even if a write tool somehow appeared in the model's context, the `CopilotModeEnforcer` intercepts it and returns a block message instead of executing it. The agent then incorporates the block message into its response, redirecting the human.

In [11]:
print("=" * 65)
print("ENFORCER — read-only passes through, write tools intercepted")
print("=" * 65)
print()

attempted_calls = [
    ("search_knowledge_base", {"query": "rollback procedure"}),
    ("restart_service",       {"service_name": "api-gateway"}),
    ("query_database",        {"sql_query": "SELECT * FROM users LIMIT 5"}),
    ("send_email",            {"to": "team@example.com", "subject": "Heads up", "body": "..."}),
]

for tool_name, args in attempted_calls:
    intercepted = enforcer.check(tool_name, args)
    status = "INTERCEPTED" if intercepted else "PERMITTED"
    print(f"  {status:11s} — {tool_name}")
    if intercepted:
        print(f"               {intercepted['message'][:80]}...")

ENFORCER — read-only passes through, write tools intercepted

  PERMITTED   — search_knowledge_base
  INTERCEPTED — restart_service
               'restart_service' is a write operation. Copilot Mode does not execute write acti...
  PERMITTED   — query_database
  INTERCEPTED — send_email
               'send_email' is a write operation. Copilot Mode does not execute write actions a...


## When to use copilot mode

| Scenario | Why copilot mode fits |
|----------|----------------------|
| Domain requires human judgment | Legal, medical, financial - the professional must own consequential actions |
| Building trust with a new agent | Demonstrate value before granting autonomous write access |
| Cost of errors is high | Automated write mistakes in regulated industries may exceed the benefit of speed |
| Users are learning from the agent | Copilot mode *teaches* rather than replaces — humans see exactly what to run |
| Audit requirements mandate human action | Regulations require a human, not an automated system, to perform write operations |

## Logging in copilot mode

| What to log | Why |
|-------------|-----|
| All read-only tool calls (name, query, result) | Audit trail of information gathered |
| All write tool interceptions (tool, prepared parameters) | Record of what was prepared and handed to the human |
| All recommendations produced | Quality review and accountability |
| Session start/end timestamps | Usage tracking |

Copilot mode does not log the human's execution of write actions - those occur outside the agent's context and are the human's own audit responsibility.

**Every response must end with an `ACTION REQUIRED` section when a write is needed.** The human should never have to guess what to do - the agent's job is to make the required human action completely unambiguous.

**Copilot mode is an excellent trust-building pattern.** Start here and escalate to supervised mode only after observing that the agent's prepared actions are consistently correct and safe.